In [2]:
import pandas as pd
import glob


files = sorted(glob.glob("../data/raw/weather_*.csv"))
print(files)

weather = pd.concat((pd.read_csv(f)for f in files), ignore_index=True)
print(weather.shape)
print(weather.columns.tolist())

['../data/raw\\weather_2017.csv', '../data/raw\\weather_2018.csv', '../data/raw\\weather_2019.csv', '../data/raw\\weather_2020.csv', '../data/raw\\weather_2021.csv', '../data/raw\\weather_2022.csv', '../data/raw\\weather_2023.csv', '../data/raw\\weather_2024.csv', '../data/raw\\weather_2025.csv', '../data/raw\\weather_2026.csv']
(3652, 31)
['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID', 'Date/Time', 'Year', 'Month', 'Day', 'Data Quality', 'Max Temp (°C)', 'Max Temp Flag', 'Min Temp (°C)', 'Min Temp Flag', 'Mean Temp (°C)', 'Mean Temp Flag', 'Heat Deg Days (°C)', 'Heat Deg Days Flag', 'Cool Deg Days (°C)', 'Cool Deg Days Flag', 'Total Rain (mm)', 'Total Rain Flag', 'Total Snow (cm)', 'Total Snow Flag', 'Total Precip (mm)', 'Total Precip Flag', 'Snow on Grnd (cm)', 'Snow on Grnd Flag', 'Dir of Max Gust (10s deg)', 'Dir of Max Gust Flag', 'Spd of Max Gust (km/h)', 'Spd of Max Gust Flag']


In [3]:
weather[["Station Name", "Climate ID"]].drop_duplicates()

,Station Name,Climate ID
0,CALGARY INTL A,3031092


In [4]:
weather = weather.rename(columns={
    "Date/Time": "date",
    "Max Temp (°C)": "max_temp",
    "Min Temp (°C)": "min_temp",
    "Mean Temp (°C)": "mean_temp",
    "Total Snow (cm)": "total_snow",
    "Total Snow Flag": "snow_flag",
    "Total Precip (mm)": "total_precip",
    "Snow on Grnd (cm)": "snow_on_ground",
})
weather = weather[["date", "max_temp", "min_temp", "mean_temp",
                   "total_snow", "snow_flag", "total_precip", "snow_on_ground"]].copy()
weather.head()

,date,max_temp,min_temp,mean_temp,total_snow,snow_flag,total_precip,snow_on_ground
0,2017-01-01,-7.8,-17.4,-12.6,3.6,NaN,2.0,12.0
1,2017-01-02,-16.1,-26.1,-21.1,0.0,T,0.0,14.0
2,2017-01-03,-15.5,-22.4,-19.0,0.4,NaN,0.3,14.0
3,2017-01-04,-7.2,-17.2,-12.2,0.0,T,0.0,13.0
4,2017-01-05,-3.5,-15.8,-9.7,2.2,NaN,1.2,13.0


In [5]:
weather["date"] = pd.to_datetime(weather["date"])

num_cols = ["max_temp", "min_temp", "mean_temp", "total_snow", "total_precip", "snow_on_ground"]
weather[num_cols] = weather[num_cols].apply(pd.to_numeric, errors="coerce")
weather.dtypes

date              datetime64[ns]
max_temp                 float64
min_temp                 float64
mean_temp                float64
total_snow               float64
snow_flag                 object
total_precip             float64
snow_on_ground           float64
dtype: object

In [6]:
print("Before:", len(weather))
weather = weather.dropna(subset=num_cols, how="all")
print("After:", len(weather))
print(weather["date"].min(), weather["date"].max())

Before: 3652
After: 3577
2017-01-01 00:00:00 2026-10-25 00:00:00


In [7]:
weather["snow_flag"].value_counts(dropna=False)

snow_flag
NaN    3220
T       304
M        53
Name: count, dtype: int64

In [8]:
weather = weather[~weather["date"].dt.month.isin([5, 6, 7, 8, 9])]

In [9]:
weather = weather.drop_duplicates(subset="date").sort_values("date")
weather["date"] = weather["date"].dt.date      # match the traffic table's date type

print(weather.shape)
print(weather[num_cols].describe().round(1))
print("Big snow days (≥ 5 cm):", (weather["total_snow"] >= 5).sum())

(2052, 8)
       max_temp  min_temp  mean_temp  total_snow  total_precip  snow_on_ground
count    2010.0    2004.0     2001.0      2023.0        2048.0          1422.0
mean        4.0      -7.5       -1.8         0.7           0.7             5.4
std        10.1       8.3        8.9         2.2           2.2             5.4
min       -30.4     -36.2      -33.0         0.0           0.0             0.0
25%        -1.6     -12.2       -6.7         0.0           0.0             1.0
50%         5.3      -5.5       -0.1         0.0           0.0             4.0
75%        10.7      -1.6        4.5         0.0           0.2             8.0
max        26.0       9.9       16.7        32.8          23.7            31.0
Big snow days (≥ 5 cm): 87
